# 🌪️ GEDA: Graph-Enhanced Differential Attention
## Multimodal Crisis Classification on CrisisMMD v2.0

**Kiến trúc:** CLIP (frozen) → PCA → Projection → GraphSAGE → Self-Attn → GCA → DiffAttn → Multi-task Heads

**Notebook này chạy toàn bộ pipeline GEDA:**
1. Setup Environment & Mount Drive
2. Download/Load Dataset CrisisMMD v2.0
3. Import GEDA code
4. Sanity Check (forward pass + loss test)
5. Quick Test — 1 run duy nhất
6. Full Experiments — 60 runs (5 seeds × 3 tasks × 4 few-shot)
7. Ablation Study — 5 biến thể kiến trúc
8. Results Visualization
9. Export kết quả & Comparison-ready CSV

---

### So sánh với các notebook khác
| Notebook | Mô hình | Vai trò |
|:---------|:--------|:--------|
| `mm_class_experiment.ipynb` | Paper 1 — GNN Semi-supervised | Baseline GNN gốc |
| `new_colab_notebook.ipynb` | Paper 2 — Differential Attention gốc | Baseline DiffAttn gốc |
| **`geda_notebook.ipynb`** | **GEDA = GNN + DiffAttn kết hợp** | **Notebook này** |
| `causal_crisis_notebook.ipynb` | CausalCrisis = GEDA + Causal Inference | Mở rộng nâng cao |

---
# 🛠️ PHASE 1: SETUP ENVIRONMENT

> ⚠️ **Chạy cell này ĐẦU TIÊN.** Khởi tạo tất cả imports, biến, và paths.

In [ ]:
#@title 1.1 Setup toàn bộ environment { display-mode: "form" }
#@markdown **Chọn nơi lưu kết quả:**
SOURCE_MODE = "download" #@param ["download", "drive"]
#@markdown - `download`: tải dataset bằng wget + lưu results trên Colab
#@markdown - `drive`: tải dataset bằng wget + mount Drive lưu results

import torch
import time
import os
import sys
import shutil
import numpy as np
from pathlib import Path

# === GPU Check ===
print("="*60)
print("  PHASE 1: Environment Setup")
print("="*60)
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Memory: {mem_gb:.1f} GB")
else:
    print("⚠️ No GPU! Runtime > Change runtime type > T4 GPU")

# === Paths (khởi tạo mặc định) ===
DATASET = None
DRIVE_DIR = None
LOCAL_RESULTS = "/content/geda_results"
RESULTS_CSV = f"{LOCAL_RESULTS}/all_results.csv"
DRIVE_RESULTS = LOCAL_RESULTS  # default
DRIVE_CKPTS = "/content/geda_checkpoints"

if SOURCE_MODE == "drive":
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = "/content/drive/MyDrive/CrisisSummarization"
    DRIVE_RESULTS = f"{DRIVE_DIR}/geda_results"
    DRIVE_CKPTS = f"{DRIVE_DIR}/geda_checkpoints"
    RESULTS_CSV = f"{DRIVE_RESULTS}/all_results.csv"
    print(f"\n📂 Mode: Download (wget) + Save to Drive")
    print(f"   Drive dir: {DRIVE_DIR}")
else:
    print(f"\n📂 Mode: Download (wget) + Save địa phương")

# === Create directories ===
for d in [DRIVE_RESULTS, DRIVE_CKPTS, LOCAL_RESULTS]:
    os.makedirs(d, exist_ok=True)

print(f"\n  ✅ Setup complete! (mode={SOURCE_MODE})")

## 1.2 Cài đặt Dependencies

In [ ]:
# Cài dependencies — faiss-gpu tự detect CUDA version
import subprocess, sys
def _install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# Core packages
_install('open_clip_torch')
_install('scikit-learn')
_install('pandas')
_install('matplotlib')
_install('seaborn')

# FAISS — ưu tiên GPU cho hiệu năng tốt nhất
try:
    import torch
    cuda_ver = torch.version.cuda  # e.g. '12.4'
    if cuda_ver and cuda_ver.startswith('12'):
        _install('faiss-gpu-cu12')
        print(f'  ✅ faiss-gpu-cu12 (CUDA {cuda_ver})')
    elif cuda_ver and cuda_ver.startswith('11'):
        _install('faiss-gpu-cu11')
        print(f'  ✅ faiss-gpu-cu11 (CUDA {cuda_ver})')
    else:
        _install('faiss-cpu')
        print(f'  ⚠️ faiss-cpu (CUDA {cuda_ver} không hỗ trợ GPU build)')
except Exception as e:
    _install('faiss-cpu')
    print(f'  ⚠️ faiss-cpu (fallback: {e})')

import faiss
ngpus = faiss.get_num_gpus() if hasattr(faiss, 'get_num_gpus') else 0
print(f'  FAISS version: {faiss.__version__}, GPUs: {ngpus}')

---
# 📦 PHASE 2: DOWNLOAD DATASET

Tải **CrisisMMD v2.0** từ CrisisNLP bằng `wget` — **giống cách repo GNN gốc** ([jdnascim/mm-class-for-disaster-data-with-gnn](https://github.com/jdnascim/mm-class-for-disaster-data-with-gnn)).

> **Luôn dùng wget** để tải dataset, bất kể SOURCE_MODE là gì.

In [ ]:
import glob as _glob

# URL chính thức từ CrisisNLP (giống repo GNN)
DATASET_URL = 'https://crisisnlp.qcri.org/data/crisismmd/CrisisMMD_v2.0.tar.gz'
DATA_DIR = '/content/datasets'
TARGET_DIR = os.path.join(DATA_DIR, 'CrisisMMD_v2.0')
TAR_PATH = os.path.join(DATA_DIR, 'CrisisMMD_v2.0.tar.gz')

# === Bước 1: Tìm dataset đã có sẵn (cache) ===
search_paths = [
    TARGET_DIR,
    '/content/CrisisMMD_v2.0',
    '/content/data/CrisisMMD_v2.0',
]
if DRIVE_DIR:
    search_paths += [
        f'{DRIVE_DIR}/CrisisMMD_v2.0',
        '/content/drive/MyDrive/CrisisMMD_v2.0',
    ]

for p in search_paths:
    if os.path.isdir(p):
        DATASET = p
        break

# === Bước 2: Luôn dùng wget nếu chưa có ===
if DATASET is not None:
    print(f'  ✅ Dataset đã có (cache): {DATASET}')

else:
    # Download bằng wget (giống repo GNN) — ưu tiên hơn Drive
    os.makedirs(DATA_DIR, exist_ok=True)
    if not os.path.exists(TAR_PATH) or os.path.getsize(TAR_PATH) < 1.8e9:
        print(f'  📥 Downloading CrisisMMD v2.0 (~2GB)...')
        print(f'     URL: {DATASET_URL}')
        !wget -q --show-progress -c -O {TAR_PATH} {DATASET_URL}
    else:
        print(f'  ✅ Archive exists: {os.path.getsize(TAR_PATH)/1e9:.2f} GB')
    
    # Giải nén bằng system tar (nhanh hơn Python tarfile)
    print('  📦 Extracting...')
    !cd {DATA_DIR} && tar xzf CrisisMMD_v2.0.tar.gz
    DATASET = TARGET_DIR
    print(f'  ✅ Dataset ready: {DATASET}')

# === Bước 3: Verify ===
if DATASET and os.path.isdir(DATASET):
    assert os.path.isdir(DATASET), f'Dataset not found: {DATASET}'
    _n_img = len(_glob.glob(os.path.join(DATASET, '**', '*.jpg'), recursive=True))
    _n_img += len(_glob.glob(os.path.join(DATASET, '**', '*.png'), recursive=True))
    _tsv_dir = os.path.join(DATASET, 'crisismmd_datasplit_all')
    print(f'\n  📊 Dataset verified:')
    print(f'     Path:   {DATASET}')
    print(f'     Images: {_n_img:,}')
    if os.path.isdir(_tsv_dir):
        print(f'     TSVs:   {len(os.listdir(_tsv_dir))} files')
else:
    print(f'  ❌ Dataset chưa sẵn sàng!')

---
# 📥 PHASE 3: IMPORT GEDA CODE

**Tự động:** tìm trên Drive hoặc tải từ GitHub

In [ ]:
sys.path.insert(0, '/content')

GITHUB_REPO = 'https://raw.githubusercontent.com/Raghvendra-14/A-Multimodal-Approach-and-Dataset-to-Crisis-Summarization-in-Tweets/main'
required_files = ['src/models/geda_model.py', 'src/trainers/geda_trainer.py']
missing = []

for f in required_files:
    dst = f'/content/{f}'
    
    if os.path.exists(dst):
        print(f"  ✅ {f} — already in /content/")
        continue
    
    # Option 1: copy từ Drive
    if DRIVE_DIR:
        drive_src = f'{DRIVE_DIR}/{f}'
        if os.path.exists(drive_src):
            shutil.copy2(drive_src, dst)
            print(f"  ✅ {f} — copied from Drive")
            continue
    
    # Option 2: tải từ GitHub bằng wget
    missing.append(f)

if missing:
    print(f"\n  📥 Downloading missing files from GitHub...")
    for f in missing:
        url = f'{GITHUB_REPO}/{f}'
        dst = f'/content/{f}'
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        !wget -q -O {dst} {url}
        if os.path.exists(dst) and os.path.getsize(dst) > 100:
            print(f"  ✅ {f} — downloaded from GitHub")
        else:
            print(f"  ❌ {f} — download failed! Check URL: {url}")

# === Verify imports ===
try:
    from src.models.geda_model import (
        GEDAModel, GEDALoss,
        SelfAttention, GuidedCrossAttention, AdaptiveDiffAttention,
    )
    from src.trainers.geda_trainer import (
        GEDATrainer,
        extract_clip_features, apply_pca, build_knn_graph,
        run_geda_experiment, run_geda_all_experiments,
    )
    print("\n  ✅ All imports successful!")
except ImportError as e:
    print(f"\n  ❌ Import failed: {e}")
    print("     Kiểm tra geda_model.py và geda_trainer.py")

---
# 🧪 PHASE 4: SANITY CHECK

## 4.1 Test Forward Pass (dữ liệu giả)

In [ ]:
import torch.nn.functional as F

B = 100

model = GEDAModel(
    img_dim=256, txt_dim=256, hidden_dim=512,
    dropout=0.3,
    use_graph=True,
    use_attention=True,
    use_mtl=True,
).to(device)

img = torch.randn(B, 256).to(device)
txt = torch.randn(B, 256).to(device)
adj = torch.eye(B).to(device)

with torch.no_grad():
    outputs = model(img, txt, adj, adj, task="task1")

print("="*60)
print("  SANITY CHECK: Forward Pass")
print("="*60)
for k, v in outputs.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:25s} shape={str(v.shape):15s} dtype={v.dtype}")

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📊 Parameters:")
print(f"  Total:     {total:>12,}")
print(f"  Trainable: {trainable:>12,}")
print(f"\n  ✅ Forward pass PASSED!")

## 4.2 Test Loss Function

In [ ]:
criterion = GEDALoss(
    task_weights=(0.4, 0.3, 0.3), gamma=2.0
).to(device)

targets = {
    "task1": torch.randint(0, 2, (B,)).to(device),
    "task2": torch.randint(0, 6, (B,)).to(device),
    "task3": torch.randint(0, 3, (B,)).to(device),
}
mask = torch.zeros(B, dtype=torch.bool).to(device)
mask[:50] = True

outputs = model(img, txt, adj, adj, task="task1")
total_loss, loss_dict = criterion(outputs, targets, mask)

print("="*60)
print("  SANITY CHECK: Loss Function")
print("="*60)
for k, v in loss_dict.items():
    val = v.item() if isinstance(v, torch.Tensor) else v
    print(f"  {k:20s}: {val:.6f}")

print(f"\n  ✅ Loss computation PASSED!")

del model, criterion, outputs
torch.cuda.empty_cache()

---
# 🔬 PHASE 5: QUICK TEST — 1 Run Thật

## 5.1 Extract CLIP Features

In [ ]:
print("="*60)
print("  PHASE 5: Feature Extraction")
print("="*60)

for task in ["task1"]:
    for split in ["train", "test"]:
        print(f"\n--- {task}/{split} ---")
        img_f, txt_f, labels = extract_clip_features(
            DATASET, task=task, split=split, device=device
        )
        print(f"  img: {img_f.shape}, txt: {txt_f.shape}")
        print(f"  labels: {len(set(labels))} unique classes")

## 5.2 Chạy 1 Experiment (task1, N=250, seed=42)

In [ ]:
print("="*60)
print("  PHASE 5: Quick Test — 1 Run")
print("="*60)

t_start = time.time()

result = run_geda_experiment(
    dataset_path=DATASET,
    task="task1",
    seed=42,
    n_labeled=250,
    k=16,
    hidden_dim=512,
    pca_dim=256,
    max_epochs=100,
    patience=30,
    lr=1e-4,
    device=device,
    results_csv=f"{LOCAL_RESULTS}/quick_test.csv",
    use_graph=True,
    use_attention=True,
    use_mtl=True,
    variant_name="geda_quicktest",
)

elapsed = time.time() - t_start

print(f"\n{'='*60}")
print(f"  QUICK TEST RESULT")
print(f"{'='*60}")
print(f"  Weighted F1: {result['weighted_f1']:.4f}")
print(f"  Macro F1:    {result['macro_f1']:.4f}")
print(f"  Best epoch:  {result.get('best_epoch', 'N/A')}")
print(f"  Time:        {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"{'='*60}")

if result['weighted_f1'] > 0.5:
    print("  ✅ Model converges! Ready for full experiments.")
else:
    print("  ⚠️ F1 thấp — kiểm tra hyperparameters.")

---
# 🚀 PHASE 6: FULL EXPERIMENTS — 60 Runs

**Cấu hình:** 5 seeds × 3 tasks × 4 few-shot sizes = 60 runs  
**Ước tính thời gian:** ~3-6 giờ trên Colab T4/L4  
**Kết quả auto-save** (Drive hoặc local)

In [ ]:
print("="*60)
print("  PHASE 6: Full GEDA Experiments")
print("="*60)

t_start = time.time()

run_geda_all_experiments(
    dataset_path=DATASET,
    seeds=(42, 123, 456, 789, 1024),
    tasks=("task1", "task2", "task3"),
    few_shot_sizes=(50, 100, 250, 500),
    device=device,
    results_csv=RESULTS_CSV,
)

elapsed = time.time() - t_start
print(f"\n  ⏱️ Total time: {elapsed/3600:.1f} hours")

if os.path.exists(RESULTS_CSV):
    backup = RESULTS_CSV.replace('.csv', f'_backup_{int(time.time())}.csv')
    shutil.copy2(RESULTS_CSV, backup)
    print(f"  💾 Backup: {backup}")

---
# 📊 PHASE 7: ABLATION STUDY

4 biến thể kiến trúc GEDA:
- **A1:** CLIP + Linear (baseline thuần)
- **A2:** CLIP + GNN only (không attention)
- **A3:** CLIP + Attention only (không GNN)
- **A4:** CLIP + GNN + Attention
- **Full GEDA** ⭐ (đã chạy Phase 6)

In [ ]:
print("="*60)
print("  PHASE 7: Ablation Study")
print("="*60)

ABLATION_CSV = f"{DRIVE_RESULTS}/ablation_results.csv"

ablation_configs = {
    "A1_clip_linear": {"use_graph": False, "use_attention": False, "use_mtl": True},
    "A2_gnn_only":    {"use_graph": True,  "use_attention": False, "use_mtl": True},
    "A3_attn_only":   {"use_graph": False, "use_attention": True,  "use_mtl": True},
    "A4_gnn_attn":    {"use_graph": True,  "use_attention": True,  "use_mtl": True},
}

for variant, config in ablation_configs.items():
    print(f"\n{'─'*40}")
    print(f"  Variant: {variant}")
    print(f"{'─'*40}")

    for task in ["task1", "task2", "task3"]:
        for seed in [42, 123, 456, 789, 1024]:
            try:
                run_geda_experiment(
                    dataset_path=DATASET,
                    task=task, seed=seed, n_labeled=250,
                    device=device,
                    results_csv=ABLATION_CSV,
                    variant_name=variant,
                    **config,
                )
            except Exception as e:
                print(f"  FAILED {variant}/{task}/seed{seed}: {e}")

print(f"\n  ✅ Ablation complete! Results: {ABLATION_CSV}")

In [ ]:
# === Ablation Results Table ===
import pandas as pd

all_dfs = []
if os.path.exists(ABLATION_CSV):
    all_dfs.append(pd.read_csv(ABLATION_CSV))
if os.path.exists(RESULTS_CSV):
    df_full = pd.read_csv(RESULTS_CSV)
    df_full_250 = df_full[df_full['few_shot'] == 250].copy()
    all_dfs.append(df_full_250)

if all_dfs:
    df_abl = pd.concat(all_dfs, ignore_index=True)

    print("\n" + "="*75)
    print("  ABLATION TABLE: Weighted F1 @ N=250")
    print("="*75)

    variant_order = ["A1_clip_linear", "A2_gnn_only", "A3_attn_only",
                     "A4_gnn_attn", "geda_full"]
    variant_labels = {
        "A1_clip_linear": "A1: CLIP + Linear",
        "A2_gnn_only":    "A2: + GNN only",
        "A3_attn_only":   "A3: + Attention only",
        "A4_gnn_attn":    "A4: + GNN + Attn",
        "geda_full":      "Full GEDA ⭐",
    }

    print(f"\n{'Variant':40s} {'Task1':>10s} {'Task2':>10s} {'Task3':>10s}")
    print("-" * 75)
    for v in variant_order:
        label = variant_labels.get(v, v)
        row = f"{label:40s}"
        for task in ["task1", "task2", "task3"]:
            mask = (df_abl['model'] == v) & (df_abl['task'] == task)
            vals = df_abl.loc[mask, 'weighted_f1'].values * 100
            if len(vals) > 0:
                row += f"  {vals.mean():.1f}±{vals.std():.1f}"
            else:
                row += f"       N/A"
        print(row)
else:
    print("  ⚠️ No results. Run Phase 6 and 7 first.")

---
# 📈 PHASE 8: RESULTS VISUALIZATION

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if os.path.exists(RESULTS_CSV):
    df = pd.read_csv(RESULTS_CSV)
    task_names = {"task1": "Informativeness", "task2": "Humanitarian", "task3": "Damage"}

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('GEDA: Weighted F1 vs Few-shot Size', fontsize=14, fontweight='bold')

    for idx, task in enumerate(["task1", "task2", "task3"]):
        ax = axes[idx]
        task_df = df[df['task'] == task]
        grouped = task_df.groupby('few_shot')['weighted_f1'].agg(['mean', 'std']).reset_index()
        grouped['mean'] *= 100; grouped['std'] *= 100

        ax.errorbar(grouped['few_shot'], grouped['mean'], yerr=grouped['std'],
                    marker='o', capsize=5, linewidth=2, markersize=8,
                    color='#1565c0', label='GEDA')
        ax.fill_between(grouped['few_shot'],
                        grouped['mean'] - grouped['std'],
                        grouped['mean'] + grouped['std'],
                        alpha=0.2, color='#1565c0')
        ax.set_title(task_names[task], fontsize=12, fontweight='bold')
        ax.set_xlabel('# Labeled Samples')
        ax.set_ylabel('Weighted F1 (%)')
        ax.set_xticks([50, 100, 250, 500])
        ax.grid(True, alpha=0.3)
        ax.legend()

    plt.tight_layout()
    save_path = f"{DRIVE_RESULTS}/geda_f1_vs_fewshot.png"
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  💾 Saved: {save_path}")
else:
    print("  ⚠️ Run Phase 6 first!")

In [ ]:
# === Box Plot ===
if os.path.exists(RESULTS_CSV):
    df = pd.read_csv(RESULTS_CSV)
    df['wf1_pct'] = df['weighted_f1'] * 100
    df['task_label'] = df['task'].map(task_names)

    fig, ax = plt.subplots(figsize=(12, 6))
    sns.boxplot(data=df, x='few_shot', y='wf1_pct', hue='task_label',
                palette='Set2', ax=ax)
    ax.set_title('GEDA: F1 Distribution', fontweight='bold')
    ax.set_xlabel('# Labeled Samples')
    ax.set_ylabel('Weighted F1 (%)')
    ax.legend(title='Task')
    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    save_path = f"{DRIVE_RESULTS}/geda_boxplot.png"
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  💾 Saved: {save_path}")

---
# 💾 PHASE 9: EXPORT & SUMMARY

In [ ]:
print("="*70)
print("  PHASE 9: Final Summary")
print("="*70)

if os.path.exists(RESULTS_CSV):
    df = pd.read_csv(RESULTS_CSV)
    print(f"\n  📊 GEDA Results: {len(df)} runs")
    task_names = {"task1": "Informativeness", "task2": "Humanitarian", "task3": "Damage"}

    print(f"\n{'Task':25s} {'N':>5s} {'wF1 (mean±std)':>16s} {'macF1':>16s}")
    print("-" * 65)
    for task in ["task1", "task2", "task3"]:
        for n in [50, 100, 250, 500]:
            mask = (df['task'] == task) & (df['few_shot'] == n)
            wf1 = df.loc[mask, 'weighted_f1'].values * 100
            mf1 = df.loc[mask, 'macro_f1'].values * 100 if 'macro_f1' in df.columns else np.zeros_like(wf1)
            if len(wf1) > 0:
                print(f"  {task_names[task]:23s} {n:5d}  {wf1.mean():.1f} ± {wf1.std():.1f}     "
                      f"{mf1.mean():.1f} ± {mf1.std():.1f}")

# Nếu mode=download, tải results về máy
if SOURCE_MODE == "download":
    from google.colab import files
    for f in [RESULTS_CSV]:
        if os.path.exists(f):
            files.download(f)
            print(f"  📥 Downloaded: {f}")

print(f"\n📁 Results: {RESULTS_CSV}")
print(f"📌 Kết quả sẵn sàng cho so sánh từ causal_crisis_notebook.ipynb")

print(f"\n{'='*70}")
print(f"  🎉 GEDA experiments complete!")
print(f"{'='*70}")